# ManualMate — embed chunks on Colab and measure the baseline

Run the cells from top to bottom. Before you start:

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click the **key icon** (Secrets) on the left. Add a secret named `DATABASE_URL` with your Neon connection string (the same value as in your `.env`). Turn on **Notebook access**.
3. Have `data/chunks/chunks.jsonl` from your laptop ready to upload.

The notebook keeps nothing important: the vectors go to Neon, and the results file is downloaded to your laptop.

## 1. Check the GPU
Expected: a line with `Tesla T4`.

In [ ]:
!nvidia-smi -L

## 2. Get the latest code from GitHub

In [ ]:
import os
if not os.path.exists("/content/manualmate"):
    !git clone https://github.com/furkanhf/manualmate.git /content/manualmate
%cd /content/manualmate
# fetch + reset (not pull): Colab keeps nothing of its own, and a plain pull fails when a
# results file made here was later committed from the laptop with the same name.
!git fetch -q origin
!git reset -q --hard origin/main
!git log --oneline -1

## 3. Install the libraries
Colab already has PyTorch with GPU support, so we do not install it.

In [ ]:
!pip install -q sentence-transformers==6.2.0 rank-bm25==0.2.2 "psycopg[binary]" pgvector python-dotenv
!pip install -q -e . --no-deps

## 4. Load the database secret
Expected: `DATABASE_URL loaded`. The value itself is never printed.

In [ ]:
from google.colab import userdata
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
print("DATABASE_URL loaded")

## 5. Upload chunks.jsonl
Click **Choose Files** and select `D:\\Projects\\manualmate\\data\\chunks\\chunks.jsonl`.
Expected: `2230 data/chunks/chunks.jsonl`.

In [ ]:
from google.colab import files
os.makedirs("data/chunks", exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
os.replace(name, "data/chunks/chunks.jsonl")
!wc -l data/chunks/chunks.jsonl

## 6. Embed all chunks and load them into Neon
Skip cells 5 and 6 if the chunks did not change (Neon keeps them).

The first run downloads bge-m3 (~2.3 GB, 1–2 min). Embedding takes about 1–2 min on a T4.
Expected last line: `Database now has 2230 chunks`.

In [ ]:
!python scripts/build_index.py

## 7. Measure retrieval (Recall@5 on 50 questions)
`routed` = parse the question → pick the manual → dense search in that manual.
- `--query-text issue`: search without brand and model words (default)
- `--query-text full`: search with the whole question
- `routed_hybrid`: the same, plus BM25 (RRF) inside the manual

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue
!python scripts/run_eval.py --method routed --query-text full
!python scripts/run_eval.py --method routed_hybrid --query-text issue

## 7b. Rerank (week 2)
`routed_rerank` = routed dense search finds the top 20 chunks → `bge-reranker-v2-m3` reads the question + each chunk together → new order.
- `--rerank-text full`: the reranker reads the whole question
- `--rerank-text issue`: the reranker reads the question without brand and model words

The first run downloads the reranker (~2.3 GB, 1–2 min). Compare with routed_issue: R@1 0.62, R@5 0.90, MRR 0.75.
Expected: `Loaded BAAI/bge-reranker-v2-m3 on cuda`, the scores, `rerank time per question`, and a list like `q033  2 -> 1`.

In [ ]:
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text full
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue

## 7c. Rerank + query rewriting (week 2)
`--rewrites` = search with the question + Gemini's manual words from `eval/rewrites.jsonl` (made on the laptop with `scripts/make_rewrites.py`, then committed). The reranker still reads the question only. Colab needs no Gemini key.
- line 1: dense only + rewrites (shows what the rewrites alone do)
- line 2: dense + rewrites + rerank

Compare with 7b (rr-issue): R@1 0.66, R@5 0.94, MRR 0.78. Expected first line: `Rewrites: 50 loaded`.

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue --rewrites
!python scripts/run_eval.py --method routed_rerank --query-text issue --rewrites

## 7d. Answers (week 3)
Full pipeline on every test question: router → dense top 20 → rerank → top 5 → Gemini → refusal gates → citation check.
Gate 1 is OFF in this run: Gemini answers every question, and the script saves the top rerank score. The summary then tries many thresholds without new Gemini calls.

Before this cell: add **GEMINI_API_KEY** to Colab Secrets (🔑 on the left). Value only: no quotes, no `GEMINI_API_KEY=`, no extra line. Turn on *Notebook access*.

Run the quick test first (3 questions per set, ~1 min). Expected: `GEMINI_API_KEY loaded`, then lines like `q001  answer ok  score=0.9...`, then `===== SUMMARY`.

In [ ]:
!pip install -q "google-genai"
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("GEMINI_API_KEY loaded")
!python scripts/run_answer_eval.py --limit 3

Full run: 50 in-scope + 30 out-of-scope + 10 clarify questions (~80 Gemini calls, about 5–8 minutes).
Then run cell 8 to download the two new files (`..._answers.jsonl` and `..._answers_summary.json`).

In [ ]:
!python scripts/run_answer_eval.py

## 7e. Week 4: section chunks (retrieval only, no Gemini)
Run after sections 1–6 with the NEW chunks.jsonl (`--mode section`, about 4,159 chunks).
Compare with week 2: routed dense R@5 0.90; routed + rerank (issue) R@1 0.66, R@5 0.94, MRR 0.78.
Watch q033, q035, q037 (the 3 microwave misses).

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue

## 7f. Week 4 step 2b: new questions on the NEW manuals (retrieval only, no Gemini)
14 questions in `eval/text_queries_new.jsonl` (ids n001–n014), one or two per new manual. Neon must already hold the 48-manual chunks (8,134), so skip cells 5 and 6.
Run after sections 1–4. Expected: `questions: 14`, `route_in_scope 1.00`, `route_single 1.00` (all 14 route to one manual on the laptop).

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue --queries eval/text_queries_new.jsonl --name new_routed_issue
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue --queries eval/text_queries_new.jsonl --name new_routed_rerank_issue

## 7g. Week 4 step 3: Hindi/Telugu questions (cross-lingual retrieval, no Gemini, no translation)
20 questions in `eval/multilingual_queries.jsonl`: 10 Hindi (h01–h10), 10 Telugu (t01–t10); 6 native script + 4 romanized each. bge-m3 searches the English chunks with the Hindi/Telugu question directly.
Run after sections 1–4 (skip 5 and 6). Expected: `questions: 20`, `route_in_scope 1.00`, `route_single 0.95` (h02 Daikin FTKS35 matches 2 manuals, like English q003).
Compare with English: routed + rerank R@5 0.98 (50 questions), 1.00 (14 new).

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue --queries eval/multilingual_queries.jsonl --name multi_routed_issue
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue --queries eval/multilingual_queries.jsonl --name multi_routed_rerank_issue

## 7h. Week 4 step 3 fix round: Hindi/Telugu → English translation, then search
Same 20 questions as 7g, but the router, dense search and reranker read Gemini's English translation from `eval/translations.jsonl` (made on the laptop with `scripts/make_translations.py`; Colab needs no Gemini key).
Expected: `Translations: 20 loaded`, `questions: 20`, `route_single 0.95`. Compare with 7g: rerank R@5 0.85 (native 12/12, romanized 5/8).

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue --queries eval/multilingual_queries.jsonl --translations --name multi_routed_issue_tr
!python scripts/run_eval.py --method routed_rerank --query-text issue --rerank-text issue --queries eval/multilingual_queries.jsonl --translations --name multi_routed_rerank_issue_tr

## 7i. Week 4 step 4: Hindi/Telugu answers (Gemini)
Run after the 7d cells (needs GEMINI_API_KEY from the first 7d cell).
The 20 questions in `eval/multilingual_queries.jsonl`: search uses the saved English translation (as in 7h), Gemini gets the ORIGINAL question and answers in Hindi or Telugu. Only this set runs (no oos/clarify), ~20 Gemini calls, 1–2 minutes.
Expected: `Translations: 20 loaded`, lines like `h01 answer ok score=0.9…`, then `===== SUMMARY` with a `hi:` and a `te:` line. Then run cell 8 (new file `<date>_answers_multilingual_queries.jsonl`).

In [ ]:
!python scripts/run_answer_eval.py --queries eval/multilingual_queries.jsonl --translations

## 9. Week 6 step 4: run the app with the REAL search (public test link)
Run sections 1–4 first (GPU, code, libraries, DATABASE_URL). Skip 5–8.

This starts the chat UI with dense search (bge-m3) + reranker on the T4, and Gemini for the answers.
Expected (about 1–2 minutes): `Loaded BAAI/bge-m3 on cuda`, `Loaded BAAI/bge-reranker-v2-m3 on cuda`,
`ManualMate ready: search=full, 8134 chunks`, then a link like `https://xxxx.gradio.live`.

Open the link and try: *"how many modes does my Daikin AC have?"*, the Daikin noise question,
a Hindi question, a romanized one (*"mera daikin ac bhot awaaz karta hai kya karu?"*) and an off-topic one (a price).
The small line under each answer shows the time of each part (search, answer, total) on the T4.

The link works while this cell runs (up to 1 week, or until Colab disconnects). It is public: do not share it.
To stop: the ■ stop button of the cell.

In [ ]:
!pip install -q gradio==6.31.0 fastapi==0.162.5 uvicorn==0.54.0 google-genai
import os, sys
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["MANUALMATE_SEARCH"] = "full"
sys.path.insert(0, "/content/manualmate/src")   # the editable install is not seen by this kernel

from manualmate.api.service import Service
from manualmate.api.ui import build_ui

service = Service("full")
service.start()                                  # loads both models on the GPU
build_ui(lambda: service).launch(share=True, debug=True)

## 8. Download the NEW results files
Only the files that are not in Git yet (the runs of this session). The old results are already on your laptop.
Save them in `D:\\Projects\\manualmate\\eval\\results\\`, then commit them. If the browser asks "Allow multiple downloads?", click Allow.

In [ ]:
from google.colab import files   # also imported in cell 5; needed when cell 5 is skipped
import subprocess

# Files in eval/results that Git does not know yet = the new runs of this session.
new = subprocess.run(["git", "ls-files", "--others", "--exclude-standard", "eval/results"],
                     capture_output=True, text=True).stdout.split()
print(f"{len(new)} new results file(s)")
for path in sorted(new):
    print(path)
    files.download(path)